
## Patch Instruction for `AutoPipeline` Before Running MASE Pipeline

To prevent the `TypeError: object of type 'NoneType' has no len()` error, follow these steps:

1. **Open the file**: ./mase/src/chop/pipelines/auto_pipeline.py

2. **Locate line 26** inside the `AutoPipeline.__init__` method.

3. **Replace the existing line** with:

```python
self.pass_outputs = [{}] * len(self.pass_groups)

In [ ]:
# !git clone --branch jtan/dev_physical https://github.com/tanjun8802/Mase_EDGE.git 
# %cd /content
# !rm -rf Mase_EDGE/mase
# %cd Mase_EDGE
!git submodule update --init --recursive
%cd mase/
%pip install -e ".[executorch]"

In [ ]:
%cd /content/Mase_EDGE
!wget http://cs231n.stanford.edu/tiny-imagenet-200.zip
!unzip tiny-imagenet-200.zip

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision.transforms as T
import torchvision.datasets as datasets
import torchvision.models as models
from pathlib import Path

# ------------------------------
# 1️⃣ Device setup
# ------------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"

# ------------------------------
# 2️⃣ Tiny ImageNet dataloaders
# ------------------------------
transform = T.Compose([
    T.Resize(224),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406],
                [0.229, 0.224, 0.225])
])

train_dataset = datasets.ImageFolder(root="./tiny-imagenet-200/train", transform=transform)
val_dataset   = datasets.ImageFolder(root="./tiny-imagenet-200/val", transform=transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=64, shuffle=False)

# ------------------------------
# 3️⃣ Load pre-trained ResNet18
# ------------------------------
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
model.fc = nn.Linear(model.fc.in_features, 200)  # Tiny ImageNet has 200 classes
model = model.to(device)

# ------------------------------
# 4️⃣ FP32 quick fine-tune (optional)
# ------------------------------
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)
epochs = 1  # short demo

for epoch in range(epochs):
    model.train()
    running_loss = 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(imgs), labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    print(f"FP32 Epoch {epoch+1}/{epochs}, Loss: {running_loss/len(train_loader):.4f}")

def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            _, preds = model(imgs).max(1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return 100 * correct / total

acc_fp32 = evaluate(model, val_loader)
print(f"✅ FP32 Accuracy: {acc_fp32:.2f}%")

# ------------------------------
# 5️⃣ Apply CHOP PTQ
# ------------------------------
from chop import MaseGraph
import chop.passes as passes

example_inputs = torch.randn(1, 3, 224, 224).to(device)
dummy_inputs = {"x": example_inputs}

mg = MaseGraph(model)
mg, _ = passes.init_metadata_analysis_pass(mg)
mg, _ = passes.add_common_metadata_analysis_pass(mg, pass_args={"dummy_in": dummy_inputs})

# Quantization configuration INT8
quantization_config = {
    "by": "type",
    "default": {"config": {"name": None}},  # leave default layers unquantized
    "linear": {
        "config": {
            "name": "integer",
            "data_in_width": 8,
            "data_in_frac_width": 7,
            "weight_width": 8,
            "weight_frac_width": 7,
            "bias_width": 8,
            "bias_frac_width": 0,   # must be 0 to avoid TorchScript issues
        }
    },
}

mg, _ = passes.quantize_transform_pass(mg, pass_args=quantization_config)
print("✅ PTQ applied to ResNet18.")

# Convert to FX GraphModule for evaluation
ptq_model = torch.fx.GraphModule(mg.model, mg.fx_graph).to(device)

acc_ptq = evaluate(ptq_model, val_loader)
print(f"✅ PTQ Accuracy: {acc_ptq:.2f}%")

# ------------------------------
# 6️⃣ Optional: QAT-like fine-tuning (PyTorch training)
# ------------------------------
# Perform QAT-like training on the FP32 model instead of FX Graph with IntegerQuantize
qat_model = model  # use FP32 model for export
optimizer = optim.Adam(qat_model.parameters(), lr=1e-4)
epochs_qat = 1  # short demo

for epoch in range(epochs_qat):
    qat_model.train()
    running_loss = 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(qat_model(imgs), labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    print(f"QAT Epoch {epoch+1}/{epochs_qat}, Loss: {running_loss/len(train_loader):.4f}")

acc_qat = evaluate(qat_model, val_loader)
print(f"✅ QAT Accuracy: {acc_qat:.2f}%")

# ------------------------------
# 7️⃣ Export model to ONNX for Android
# ------------------------------
output_dir = Path("./mase_output")
output_dir.mkdir(exist_ok=True)

# Use FP32 QAT model for export
qat_model.eval()
torch.onnx.export(
    qat_model,
    example_inputs,
    str(output_dir / "resnet18_qat_fp32.onnx"),
    opset_version=17,
    input_names=["x"],
    output_names=["output"],
    dynamic_axes={"x": {0: "batch_size"}, "output": {0: "batch_size"}}
)
print(f"✅ ONNX FP32 QAT model exported to {output_dir / 'resnet18_qat_fp32.onnx'}")